# Text classification

Sort text into categories: sentiment, topic, intent, spam, priority.

Two ways to start, from Hello Model: https://sayhellomodel.com/models/text-classification/

**Your data:** the code uses example file names (such as `data.csv`). Upload your own file with the Files panel on the left, or change the name in the code. Run the cells from the top.

## Quick start

LLM zero-shot / few-shot classification via API: No training needed — describe the categories in a prompt. Good for prototypes or fewer than 100 examples.

In [ ]:
%pip install -q anthropic

In [ ]:
# pip install anthropic
import anthropic
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY

LABELS = ["billing", "bug", "feature_request", "other"]
def classify(text: str) -> str:
    msg = client.messages.create(
        model="claude-haiku-5-5", max_tokens=10,
        messages=[{"role": "user", "content":
            f"Classify this support ticket into one of {LABELS}. "
            f"Answer with the label only.\n\n{text}"}])
    return msg.content[0].text.strip()

print(classify("I was charged twice this month"))

## Train your own model

Fine-tuned small transformer (DistilBERT / ModernBERT) or SetFit: Cheap to run, fast, and accurate once you have a few hundred labels.

In [ ]:
%pip install -q transformers datasets evaluate accelerate

In [ ]:
# pip install transformers datasets evaluate accelerate
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

ds = load_dataset("csv", data_files={"train": "train.csv", "test": "test.csv"})
labels = sorted(set(ds["train"]["label"]))
l2id = {l: i for i, l in enumerate(labels)}
ds = ds.map(lambda x: {"labels": l2id[x["label"]]})

name = "distilbert-base-uncased"
tok = AutoTokenizer.from_pretrained(name)
ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=256), batched=True)
model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=len(labels))

args = TrainingArguments("out", num_train_epochs=3, per_device_train_batch_size=16,
                         eval_strategy="epoch", learning_rate=2e-5)
trainer = Trainer(model=model, args=args, train_dataset=ds["train"],
                  eval_dataset=ds["test"], tokenizer=tok)
trainer.train()
trainer.save_model("model")